# Bronze to Silver
### CineData Analytics
### RocketLab 2026.2

OBS -> As tabelas da camada Bronze são utilizadas apenas como fonte e não são modificadas.

In [0]:
# Configuração do ambiente
catalog = "rocketlab"

bronze_schema_name = "bronze"
silver_schema_name = "silver"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

print(f"Bronze: {bronze_schema}")
print(f"Silver: {silver_schema}")

Bronze: rocketlab.bronze
Silver: rocketlab.silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {silver_schema}"
)

print(f"Ok Schema {silver_schema} disponível.")

display(
    spark.sql(f"SHOW SCHEMAS IN {catalog}")
)

Ok Schema rocketlab.silver disponível.


databaseName
bronze
default
gold
information_schema
silver


## silver.tb_info_filmes

Transformação das informações gerais dos filmes, incluindo padronização dos nomes das colunas, conversão de datas e duração, normalização do status dos filmes e remoção de registros duplicados.

In [0]:
df_bronze_info = spark.table(
    f"{bronze_schema}.tb_movies_info"
)

display(df_bronze_info.limit(10))

df_bronze_info.printSchema()

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-20T16:29:48.170Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-20T16:29:48.170Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-20T16:29:48.170Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-20T16:29:48.170Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-20T16:29:48.170Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-09-20T16:29:48.170Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-09-20T16:29:48.170Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-09-20T16:29:48.170Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as t

root
 |-- id: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
# Mapeamento de colunas
df_info = df_bronze_info.select(
    F.col("id").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    F.col("release_date").alias("data_lancamento"),
    F.col("runtime").alias("duracao_minutos"),
    F.col("original_language").alias("idioma_original"),
    F.col("status").alias("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao"),
    F.col("ingestion_datetime")
)

# Deduplicação com versão mais recente
    # Considero a data de ingestão da camada bronze e mantenho a versão mais recente
window_filmes = (
    Window
    .partitionBy("id_filme")
    .orderBy(F.col("ingestion_datetime").desc())
)

df_info = (
    df_info
    .withColumn(
        "_row_number",
        F.row_number().over(window_filmes)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

# Normalizando coluna status_filme
df_info = df_info.withColumn(
    "status_filme",
    F.lower(
        F.trim(
            F.regexp_replace(
                F.col("status_filme"),
                r"-+", # Sequência de hífens
                " "
            )
        )
    )
)

df_info = df_info.withColumn(
    "status_filme",
    F.regexp_replace(
        F.col("status_filme"),
        r"\s+", # Sequência de espaços em branco
        " "
    )
)

# Traduzindo status_filme
df_info = df_info.withColumn(
    "status_filme",
    F.when(F.col("status_filme") == "released", "Lançado")
    .when(F.col("status_filme") == "post production", "Pós-Produção")
    .when(F.col("status_filme") == "in production", "Em Produção")
    .when(F.col("status_filme") == "planned", "Planejado")
    .when(F.col("status_filme") == "rumored", "Rumores")
    .when(F.col("status_filme") == "canceled", "Cancelado")
    .otherwise("Não Informado") # Classificando valores não mapeáveis
)

# Tratamento da data de lançamento
    # A base apresenta datas em múltiplos formatos
    # A conversão testa os padrões que eu identifiquei nos dados e mantém NULL
    # quando nenhum dos formatos conhecidos puder ser interpretado
df_info = df_info.withColumn(
    "data_lancamento",
    F.coalesce(
        F.expr("try_to_timestamp(data_lancamento, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(data_lancamento, 'dd/MM/yyyy')"),
        F.expr("try_to_timestamp(data_lancamento, 'MM-dd-yyyy')"),
        F.expr("try_to_timestamp(data_lancamento, 'dd-MM-yyyy')")
    ).cast("date")
)

# Criando coluna ano_lancamento
    # Durações incompatíveis com int são tratadas como null
df_info = (
    df_info
    .withColumn(
        "duracao_minutos",
        F.expr("try_cast(duracao_minutos AS INT)")
    )
    .withColumn(
        "ano_lancamento",
        F.year("data_lancamento")
    )
    .drop("ingestion_datetime") # Dropando a coluna de data de ingestão
)

# Gravando a tabela final
(
    df_info.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_info_filmes")
)

print(
    f"Ok -> {silver_schema}.tb_info_filmes "
    "criada com sucesso."
)

# Validando tabela 
df_info_silver = spark.table(
    f"{silver_schema}.tb_info_filmes"
)

duplicados = (
    df_info_silver
    .groupBy("id_filme")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Ok tb_info_filmes | "
    f"Registros: {df_info_silver.count()} | "
    f"IDs duplicados: {duplicados}"
)

display(
    df_info_silver
    .groupBy("status_filme")
    .count()
    .orderBy(F.desc("count"))
)

Ok -> rocketlab.silver.tb_info_filmes criada com sucesso.
Ok tb_info_filmes | Registros: 97879 | IDs duplicados: 0


status_filme,count
Lançado,96463
Pós-Produção,701
Em Produção,604
Não Informado,64
Planejado,47


## silver.tb_financeiro_filmes

Tratamento das informações financeiras dos filmes. Valores monetários inconsistentes são normalizados e convertidos para tipos numéricos, valores inválidos são tratados e são calculados orçamento, receita, lucro e margem de lucro em USD e BRL.

In [0]:
df_bronze_financeiro = spark.table(
    f"{bronze_schema}.tb_movies_financials"
)

# Mapeamento das colunas
df_financeiro = df_bronze_financeiro.select(
    F.col("id").alias("id_filme"),
    F.col("budget").alias("orcamento_usd"),
    F.col("revenue").alias("receita_usd")
)

# Normalizando os valores monetários
df_financeiro = (# Padronizando os valores
    df_financeiro

    .withColumn(
        "_orcamento_limpo",
        F.upper(F.trim(F.col("orcamento_usd")))
    )

    .withColumn(
        "_receita_limpa",
        F.upper(F.trim(F.col("receita_usd")))
    )
)

# Removendo caracteres especiais
df_financeiro = (
    df_financeiro

    .withColumn(
        "_orcamento_limpo",
        F.regexp_replace(
            F.col("_orcamento_limpo"),
            r"USD|\$|,|\s",
            ""
        )
    )

    .withColumn(
        "_receita_limpa",
        F.regexp_replace(
            F.col("_receita_limpa"),
            r"USD|\$|,|\s",
            ""
        )
    )
)

# Tratando valores em milhões (34.0M vira 34000000.00)
    # Valores textuais como "Unknown" viram NULL
df_financeiro = df_financeiro.withColumn(
    "orcamento_usd",
    F.when(
        F.col("_orcamento_limpo").rlike(
            r"^-?\d+(\.\d+)?M$"
        ),
        F.expr(
            """
            try_cast(
                regexp_replace(_orcamento_limpo, 'M', '')
                AS DECIMAL(18,2)
            ) * 1000000
            """
        )
    )
    .otherwise(
        F.expr(
            """
            try_cast(
                _orcamento_limpo
                AS DECIMAL(18,2)
            )
            """
        )
    )
)
df_financeiro = df_financeiro.withColumn(
    "receita_usd",

    F.when(
        F.col("_receita_limpa").rlike(
            r"^-?\d+(\.\d+)?M$"
        ),
        F.expr(
            """
            try_cast(
                regexp_replace(_receita_limpa, 'M', '')
                AS DECIMAL(18,2)
            ) * 1000000
            """
        )
    )
    .otherwise(
        F.expr(
            """
            try_cast(
                _receita_limpa
                AS DECIMAL(18,2)
            )
            """
        )
    )
)

# Tratando valores inválidos
    # Orçamentos e receitas <=0 são tratados como null
df_financeiro = (
    df_financeiro

    .withColumn(
        "orcamento_usd",
        F.when(
            F.col("orcamento_usd") > 0,
            F.col("orcamento_usd")
        )
    )

    .withColumn(
        "receita_usd",
        F.when(
            F.col("receita_usd") > 0,
            F.col("receita_usd")
        )
    )
)

# Remove colunas usadas durante limpeza
df_financeiro = df_financeiro.drop(
    "_orcamento_limpo",
    "_receita_limpa"
)

# Obtendo a cotação mais recente do dólar disponível na bronze
df_cotacao = (
    spark.table(
        f"{bronze_schema}.tb_cotacao_dolar"
    )
    .withColumn(
        "data_hora_cotacao",
        F.to_timestamp("dataHoraCotacao")
    )
)

cotacao_mais_recente = (
    df_cotacao
    .orderBy(
        F.col("data_hora_cotacao").desc()
    )
    .select("cotacaoCompra")
    .first()
)

if cotacao_mais_recente is None:
    raise ValueError(
        "Nenhuma cotação do dólar disponível."
    )

cotacao_dolar = cotacao_mais_recente["cotacaoCompra"]

# Convertendo de USD para BRL
df_financeiro = (
    df_financeiro

    .withColumn(
        "orcamento_brl",
        (
            F.col("orcamento_usd")
            * F.lit(cotacao_dolar)
        ).cast("decimal(18,2)")
    )

    .withColumn(
        "receita_brl",
        (
            F.col("receita_usd")
            * F.lit(cotacao_dolar)
        ).cast("decimal(18,2)")
    )
)

# Definindo lucro (receita - orçamento)
df_financeiro = (
    df_financeiro

    .withColumn(
        "lucro_usd",
        (
            F.col("receita_usd")
            - F.col("orcamento_usd")
        ).cast("decimal(18,2)")
    )

    .withColumn(
        "lucro_brl",
        (
            F.col("receita_brl")
            - F.col("orcamento_brl")
        ).cast("decimal(18,2)")
    )
)

# Definindo margem de lucro percentual
df_financeiro = df_financeiro.withColumn(
    "margem_lucro_percentual",

    F.when(
        # Evitando operações inválidas com null ou divisão por 0
        F.col("orcamento_usd").isNotNull() 
        & F.col("receita_usd").isNotNull()
        & (F.col("orcamento_usd") > 0),

        (
            (
                F.col("receita_usd")
                - F.col("orcamento_usd")
            )
            / F.col("orcamento_usd")
            * 100
        ).cast("decimal(10,2)")
    )
)

# Padronizando os tipos monetários 
df_financeiro = (
    df_financeiro
    .withColumn(
        "orcamento_usd",
        F.col("orcamento_usd").cast("decimal(20,2)")
    )
    .withColumn(
        "receita_usd",
        F.col("receita_usd").cast("decimal(20,2)")
    )
    .withColumn(
        "orcamento_brl",
        F.col("orcamento_brl").cast("decimal(20,2)")
    )
    .withColumn(
        "receita_brl",
        F.col("receita_brl").cast("decimal(20,2)")
    )
    .withColumn(
        "lucro_usd",
        F.col("lucro_usd").cast("decimal(20,2)")
    )
    .withColumn(
        "lucro_brl",
        F.col("lucro_brl").cast("decimal(20,2)")
    )
)

# df_financeiro.printSchema()
# display(df_financeiro.limit(10))

# Gravando e validando tabela final
(
    df_financeiro.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{silver_schema}.tb_financeiro_filmes"
    )
)

print(
    f"Ok -> {silver_schema}.tb_financeiro_filmes "
    "criada com sucesso."
)

df_financeiro_silver = spark.table(
    f"{silver_schema}.tb_financeiro_filmes"
)

valores_invalidos = (
    df_financeiro_silver
    .filter(
        (F.col("orcamento_usd") <= 0)
        | (F.col("receita_usd") <= 0)
    )
    .count()
)

print(
    f"Ok tb_financeiro_filmes | "
    f"Registros: {df_financeiro_silver.count()} | "
    f"Valores monetários <= 0: {valores_invalidos} | "
    f"Cotação utilizada: R$ {cotacao_dolar}"
)


Ok -> rocketlab.silver.tb_financeiro_filmes criada com sucesso.
Ok tb_financeiro_filmes | Registros: 106165 | Valores monetários <= 0: 0 | Cotação utilizada: R$ 5.1569


## silver.tb_metricas_engajamento

Padronização das métricas de popularidade, avaliações e quantidade de votos provenientes do TMDB e IMDb. Valores inconsistentes ou fora dos limites esperados são tratados antes da persistência.

In [0]:
df_bronze_metricas = spark.table(
    f"{bronze_schema}.tb_movies_metrics"
)

# Identificando registros com indícios de Column Shift
    # averageRating deve representar uma nota numérica 
    # numVotes deve representar uma quantidade inteira
condicao_desalinhamento = (
    (
        F.col("averageRating").isNotNull()
        & ~F.trim(F.col("averageRating"))
            .rlike(r"^[0-9]+([.,][0-9]+)?$")
    )
    |
    (
        F.col("numVotes").isNotNull()
        & ~F.trim(F.col("numVotes"))
            .rlike(r"^[0-9]+$")
    )
)

# Mapeamento das colunas
    # Quando há evidência de desalinhamento estrutural na linha
    # As métricas são anuladas, já que sua posição não é confiável
    # Escolhi usar null para manter o id_filme como informação
df_metricas = df_bronze_metricas.select(
    F.col("id").alias("id_filme"),

    F.when(
        ~condicao_desalinhamento,
        F.col("popularity")
    ).alias("popularidade"),

    F.when(
        ~condicao_desalinhamento,
        F.col("vote_average")
    ).alias("nota_media_tmdb"),

    F.when(
        ~condicao_desalinhamento,
        F.col("vote_count")
    ).alias("qtd_votos_tmdb"),

    F.when(
        ~condicao_desalinhamento,
        F.col("averageRating")
    ).alias("nota_media_imdb"),

    F.when(
        ~condicao_desalinhamento,
        F.col("numVotes")
    ).alias("qtd_votos_imdb")
)


# Normalizando popularidade
df_metricas = df_metricas.withColumn(
    "popularidade",
    F.regexp_replace(
        F.trim(F.col("popularidade")),
        ",",
        "."
    )
)


# Convertendo tipos as conversões incompativeis são tratadas como null
df_metricas = (
    df_metricas
    .withColumn(
        "popularidade",
        F.expr(
            "try_cast(popularidade AS DOUBLE)"
        )
    )
    .withColumn(
        "nota_media_tmdb",
        F.expr(
            "try_cast(trim(nota_media_tmdb) AS DOUBLE)"
        )
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.expr(
            "try_cast(trim(qtd_votos_tmdb) AS INT)"
        )
    )
    .withColumn(
        "nota_media_imdb",
        F.expr(
            "try_cast(trim(nota_media_imdb) AS DOUBLE)"
        )
    )
    .withColumn(
        "qtd_votos_imdb",
        F.expr(
            "try_cast(trim(qtd_votos_imdb) AS INT)"
        )
    )
)


# Tratando notas fora do intervalo [0,10]
df_metricas = (
    df_metricas
    .withColumn(
        "nota_media_tmdb",
        F.when(
            F.col("nota_media_tmdb").between(0, 10),
            F.col("nota_media_tmdb")
        )
    )
    .withColumn(
        "nota_media_imdb",
        F.when(
            F.col("nota_media_imdb").between(0, 10),
            F.col("nota_media_imdb")
        )
    )
)


# Tratando valores negativos para popularidade e quantidade de votos
df_metricas = (
    df_metricas
    .withColumn(
        "popularidade",
        F.when(
            F.col("popularidade") >= 0,
            F.col("popularidade")
        )
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.when(
            F.col("qtd_votos_tmdb") >= 0,
            F.col("qtd_votos_tmdb")
        )
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(
            F.col("qtd_votos_imdb") >= 0,
            F.col("qtd_votos_imdb")
        )
    )
)

# df_metricas.printSchema()
# display(df_metricas.limit(10))

# Gravando e validando tabela final
(
    df_metricas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{silver_schema}.tb_metricas_engajamento"
    )
)

print(
    f"Ok -> {silver_schema}.tb_metricas_engajamento "
    "criada com sucesso."
)

df_metricas_silver = spark.table(
    f"{silver_schema}.tb_metricas_engajamento"
)

valores_invalidos = (
    df_metricas_silver
    .filter(
        (~F.col("nota_media_tmdb").between(0, 10))
        | (~F.col("nota_media_imdb").between(0, 10))
        | (F.col("popularidade") < 0)
        | (F.col("qtd_votos_tmdb") < 0)
        | (F.col("qtd_votos_imdb") < 0)
    )
    .count()
)

print(
    f"Ok tb_metricas_engajamento | "
    f"Registros: {df_metricas_silver.count()} | "
    f"Valores fora dos limites: {valores_invalidos}"
)

Ok -> rocketlab.silver.tb_metricas_engajamento criada com sucesso.
Ok tb_metricas_engajamento | Registros: 107364 | Valores fora dos limites: 0


## silver.tb_avaliacoes_usuarios

Tratamento das avaliações realizadas pelos usuários, incluindo validação das notas, padronização de comentários ausentes e remoção de avaliações duplicadas.

In [0]:
df_bronze_avaliacoes = spark.table(
    f"{bronze_schema}.tb_movies_reviews"
)

# Mapeamento das colunas 
df_avaliacoes = df_bronze_avaliacoes.select(
    F.col("id").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    F.col("nota").alias("nota_usuario"),
    F.col("comentario").alias("comentario_usuario")
)

# Convertendo nota para double
df_avaliacoes = df_avaliacoes.withColumn(
    "nota_usuario",
    F.expr(
        "try_cast(trim(nota_usuario) AS DOUBLE)"
    )
)

# Tratando intervalo das notas [0,10] notas fora do intervalo viram null
df_avaliacoes = df_avaliacoes.withColumn(
    "nota_usuario",
    F.when(
        F.col("nota_usuario").between(0, 10),
        F.col("nota_usuario")
    )
)

# Tratando comentários
    # Removendo comentários vazios ou nulos viram "Sem comentário"
df_avaliacoes = df_avaliacoes.withColumn(
    "comentario_usuario",
    F.when(
        F.col("comentario_usuario").isNull()
        | (F.trim(F.col("comentario_usuario")) == ""),
        F.lit("Sem comentário")
    )
    .otherwise(
        F.trim(F.col("comentario_usuario"))
    )
)

# Removendo avaliações duplicadas onde filme,usuário, nota e comentário são iguais
df_avaliacoes = df_avaliacoes.dropDuplicates([
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
])

# df_avaliacoes.printSchema()
# display(df_avaliacoes.limit(10))

# Gravando e validando tabela final
(
    df_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{silver_schema}.tb_avaliacoes_usuarios"
    )
)

print(
    f"Ok -> {silver_schema}.tb_avaliacoes_usuarios "
    "criada com sucesso."
)

df_avaliacoes_silver = spark.table(
    f"{silver_schema}.tb_avaliacoes_usuarios"
)

notas_invalidas = (
    df_avaliacoes_silver
    .filter(
        (F.col("nota_usuario") < 0)
        | (F.col("nota_usuario") > 10)
    )
    .count()
)

duplicados = (
    df_avaliacoes_silver
    .groupBy(
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Ok tb_avaliacoes_usuarios | "
    f"Registros: {df_avaliacoes_silver.count()} | "
    f"Notas fora de 0-10: {notas_invalidas} | "
    f"Duplicados: {duplicados}"
)

Ok -> rocketlab.silver.tb_avaliacoes_usuarios criada com sucesso.
Ok tb_avaliacoes_usuarios | Registros: 32412 | Notas fora de 0-10: 0 | Duplicados: 0


## silver.tb_generos

Normalização dos gêneros associados aos filmes. Os múltiplos gêneros armazenados em uma mesma coluna são separados em registros individuais, resíduos inválidos são removidos e relações duplicadas entre filme e gênero são eliminadas.


In [0]:
df_bronze_creditos = spark.table(
    f"{bronze_schema}.tb_credits_and_tags"
)

# Mapeando as colunas
df_generos = df_bronze_creditos.select(
    F.col("id").alias("id_filme"),
    F.col("genres").alias("generos")
)

# Normalizando separadores e explosão de gêneros
    # A base possui múltiplos gêneros na mesma coluna e inconsistências
    # nos separadores, os valores são separados para que cada registro represente um único gênero associado a um filme
df_generos = (
    df_generos
    .withColumn(
        "generos",
        F.regexp_replace(
            F.col("generos"),
            r"[;|]",
            ","
        )
    )
    .withColumn(
        "genero",
        F.explode(
            F.split(F.col("generos"), ",")
        )
    )
    .select(
        "id_filme",
        F.trim(F.col("genero")).alias("genero")
    )
)

# A análise exploratória da Bronze identificou os gêneros válidos presentes na base valores fora desse domínio, são resíduos de column shift.
generos_validos = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western"
]

df_generos = df_generos.filter(
    F.col("genero").isin(generos_validos)
)

# Removendo duplicatas entre filme e gênero
df_generos = df_generos.dropDuplicates([
    "id_filme",
    "genero"
])

# df_generos.printSchema()
# display(
#     df_generos
#     .orderBy("id_filme", "genero")
#     .limit(10)
# )

# Gravando e validando tabela final
(
    df_generos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{silver_schema}.tb_generos"
    )
)

print(
    f"ok -> {silver_schema}.tb_generos "
    "criada com sucesso."
)

df_generos_silver = spark.table(
    f"{silver_schema}.tb_generos"
)

duplicados = (
    df_generos_silver
    .groupBy("id_filme", "genero")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"[OK] tb_generos | "
    f"Registros: {df_generos_silver.count()} | "
    f"Gêneros distintos: "
    f"{df_generos_silver.select('genero').distinct().count()} | "
    f"Duplicados: {duplicados}"
)

ok -> rocketlab.silver.tb_generos criada com sucesso.
[OK] tb_generos | Registros: 141965 | Gêneros distintos: 19 | Duplicados: 0


## silver.tb_pessoas_empresas

Consolidação das entidades relacionadas aos filmes em uma estrutura única.

OBS -> São considerados quatro tipos de entidade:

- Ator
- Diretor
- Roteirista
- Produtora

Os valores são separados, padronizados e classificados de acordo com seu tipo de atuação, com remoção de registros inválidos e duplicados.

In [0]:
df_bronze_creditos = spark.table(
    f"{bronze_schema}.tb_credits_and_tags"
)


# Mapeamento e dimensão unificada
    # Consolido atores, roteiristas e produtoras em uma estrutura única

df_atores = df_bronze_creditos.select(
    F.col("id").alias("id_filme"),
    F.col("cast").alias("nome_entidade")
).withColumn(
    "tipo_entidade",
    F.lit("Ator")
)

df_diretores = df_bronze_creditos.select(
    F.col("id").alias("id_filme"),
    F.col("directors").alias("nome_entidade")
).withColumn(
    "tipo_entidade",
    F.lit("Diretor")
)

df_roteiristas = df_bronze_creditos.select(
    F.col("id").alias("id_filme"),
    F.col("writers").alias("nome_entidade")
).withColumn(
    "tipo_entidade",
    F.lit("Roteirista")
)

df_produtoras = df_bronze_creditos.select(
    F.col("id").alias("id_filme"),
    F.col("production_companies").alias("nome_entidade")
).withColumn(
    "tipo_entidade",
    F.lit("Produtora")
)

df_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
)


# Desmembrando múltiplos valores e limpando
    # Cada registro passa a representar uma única entidade por filme
df_pessoas_empresas = (
    df_pessoas_empresas
    .withColumn(
        "nome_entidade",
        F.explode(
            F.split(F.col("nome_entidade"), ",")
        )
    )
    .withColumn(
        "nome_entidade",
        F.trim(
            F.regexp_replace(
                F.trim(F.col("nome_entidade")),
                '^"+|"+$',
                ""
            )
        )
    )
)

# Removendo valores vazios, column shift e marcadores de ausência
marcadores_ausencia = [
    "n/a",
    "na",
    "null",
    "none",
    "unknown",
    "not available",
    "não informado"
]

df_pessoas_empresas = (
    df_pessoas_empresas
    .filter(
        F.col("nome_entidade").isNotNull()
        & (F.trim(F.col("nome_entidade")) != "")
        & F.col("nome_entidade").rlike(r"[\p{L}\p{N}]")
        & ~F.lower(F.trim(F.col("nome_entidade"))).isin(
            marcadores_ausencia
        )
    )
)

df_pessoas_empresas = df_pessoas_empresas.filter(
    ~F.col("nome_entidade").rlike(
        r"^[+-]?\d+([.,]\d+)?$"
    )
)

df_pessoas_empresas = df_pessoas_empresas.filter(
    ~F.col("nome_entidade").contains(";")
)

# Padronizando capitalização
df_pessoas_empresas = df_pessoas_empresas.withColumn(
    "nome_entidade",
    F.initcap(
        F.lower(
            F.col("nome_entidade")
        )
    )
)

# Removendo duplicatas
df_pessoas_empresas = (
    df_pessoas_empresas
    .dropDuplicates([
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    ])
)

# df_pessoas_empresas.printSchema()

# Gravando e validando tabela final
(
    df_pessoas_empresas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{silver_schema}.tb_pessoas_empresas"
    )
)

print(
    f"Ok -> {silver_schema}.tb_pessoas_empresas "
    "criada com sucesso."
)

df_pessoas_empresas_silver = spark.table(
    f"{silver_schema}.tb_pessoas_empresas"
)

duplicados = (
    df_pessoas_empresas_silver
    .groupBy(
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Ok tb_pessoas_empresas | "
    f"Registros: {df_pessoas_empresas_silver.count()} | "
    f"Duplicados: {duplicados}"
)

display(
    df_pessoas_empresas_silver
    .groupBy("tipo_entidade")
    .count()
    .orderBy("tipo_entidade")
)

Ok -> rocketlab.silver.tb_pessoas_empresas criada com sucesso.
Ok tb_pessoas_empresas | Registros: 897550 | Duplicados: 0


tipo_entidade,count
Ator,547169
Diretor,102813
Produtora,120237
Roteirista,127331


## silver.tb_cotacao_dolar

Construção de uma série temporal diária da cotação do dólar.

Como não existem cotações do Banco Central em finais de semana e feriados, os dias ausentes são adicionados à série e preenchidos utilizando Forward Fill, mantendo a última cotação disponível.

In [0]:
# Mapeamento das colunas
df_cotacao = (
    spark.table(
        f"{bronze_schema}.tb_cotacao_dolar"
    )
    .select(
        F.col("dataHoraCotacao").alias("data_hora_cotacao"),
        F.col("cotacaoCompra").alias("cotacao_compra")
    )
)

# Convertendo os tipos
df_cotacao = (
    df_cotacao
    .withColumn(
        "data_cotacao",
        F.to_date(
            F.to_timestamp("data_hora_cotacao")
        )
    )
    .withColumn(
        "cotacao_compra",
        F.col("cotacao_compra").cast("double")
    )
    .select(
        "data_cotacao",
        "cotacao_compra"
    )
)

# Removendo as cotações duplicadas que vem da bronze
df_cotacao = df_cotacao.dropDuplicates([
    "data_cotacao",
    "cotacao_compra"
])

# Definindo intervalo temporal
intervalo = (
    df_cotacao
    .agg(
        F.min("data_cotacao").alias("data_inicial"),
        F.max("data_cotacao").alias("data_final")
    )
    .first()
)

data_inicial = intervalo["data_inicial"]
data_final = intervalo["data_final"]

# print(f"Data inicial: {data_inicial}")
# print(f"Data final: {data_final}")

# Série temporal contínua
    # Crio uma data para cada dia entre a primeira e a última cotação
    # Incluindo finais de semana e feriados
df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_inicial),
                F.lit(data_final),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

# Junção com as cotações disponíveis
df_cotacao_completa = (
    df_calendario
    .join(
        df_cotacao,
        on="data_cotacao",
        how="left"
    )
)

# Aplicando forward fill para dias sem cotação
janela_forward_fill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

df_cotacao_completa = (
    df_cotacao_completa
    .withColumn(
        "cotacao_compra",
        F.last(
            "cotacao_compra",
            ignorenulls=True
        ).over(janela_forward_fill)
    )
)


# Padrão final dos tipos
df_cotacao_completa = (
    df_cotacao_completa
    .select(
        F.col("data_cotacao").cast("date"),
        F.col("cotacao_compra").cast("double")
    )
)

# df_cotacao_completa.printSchema()
# display(
#     df_cotacao_completa
#     .orderBy("data_cotacao")
# )

# Gravando e validando tabela final
(
    df_cotacao_completa.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{silver_schema}.tb_cotacao_dolar"
    )
)

print(
    f"Ok -> {silver_schema}.tb_cotacao_dolar "
    "criada com sucesso."
)

df_cotacao_silver = spark.table(
    f"{silver_schema}.tb_cotacao_dolar"
)

cotacoes_nulas = (
    df_cotacao_silver
    .filter(F.col("cotacao_compra").isNull())
    .count()
)

datas_duplicadas = (
    df_cotacao_silver
    .groupBy("data_cotacao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Ok tb_cotacao_dolar | "
    f"Registros: {df_cotacao_silver.count()} | "
    f"Cotações nulas: {cotacoes_nulas} | "
    f"Datas duplicadas: {datas_duplicadas}"
)

display(
    df_cotacao_completa
    .orderBy("data_cotacao")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Ok -> rocketlab.silver.tb_cotacao_dolar criada com sucesso.
Ok tb_cotacao_dolar | Registros: 8 | Cotações nulas: 0 | Datas duplicadas: 0


data_cotacao,cotacao_compra
2026-09-11,5.0912
2026-09-12,5.0912
2026-09-13,5.0912
2026-09-14,5.169
2026-09-15,5.1484
2026-09-16,5.152
2026-09-17,5.1515
2026-09-18,5.1569


### Validacao final da camada

In [0]:
from datetime import datetime

dq_results = []

def add_dq_result(table_name, check_name, df, condition):
    total_rows = df.count()

    failed_rows = (
        df
        .filter(condition)
        .count()
    )

    dq_results.append({
        "table_name": table_name,
        "check_name": check_name,
        "total_rows": total_rows,
        "failed_rows": failed_rows,
        "passed": failed_rows == 0,
        "checked_at": datetime.now()
    })


# Ifno_filmes

df = spark.table(
    f"{silver_schema}.tb_info_filmes"
)

add_dq_result(
    "silver.tb_info_filmes",
    "id_filme não nulo",
    df,
    F.col("id_filme").isNull()
)

status_validos = [
    "Lançado",
    "Pós-Produção",
    "Em Produção",
    "Planejado",
    "Rumores",
    "Cancelado",
    "Não Informado"
]

add_dq_result(
    "silver.tb_info_filmes",
    "status_filme em domínio válido",
    df,
    ~F.col("status_filme").isin(status_validos)
)

add_dq_result(
    "silver.tb_info_filmes",
    "duracao_minutos não negativa",
    df,
    F.col("duracao_minutos") < 0
)

# Unicidade de id_filme
duplicados = (
    df
    .groupBy("id_filme")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dq_results.append({
    "table_name": "silver.tb_info_filmes",
    "check_name": "id_filme único",
    "total_rows": df.count(),
    "failed_rows": duplicados,
    "passed": duplicados == 0,
    "checked_at": datetime.now()
})

# Financeiro_filmes

df = spark.table(
    f"{silver_schema}.tb_financeiro_filmes"
)

add_dq_result(
    "silver.tb_financeiro_filmes",
    "orcamento_usd > 0 quando informado",
    df,
    F.col("orcamento_usd") <= 0
)

add_dq_result(
    "silver.tb_financeiro_filmes",
    "receita_usd > 0 quando informada",
    df,
    F.col("receita_usd") <= 0
)

# Metricas_engajamento

df = spark.table(
    f"{silver_schema}.tb_metricas_engajamento"
)

add_dq_result(
    "silver.tb_metricas_engajamento",
    "nota_media_tmdb entre 0 e 10",
    df,
    (F.col("nota_media_tmdb") < 0)
    | (F.col("nota_media_tmdb") > 10)
)

add_dq_result(
    "silver.tb_metricas_engajamento",
    "nota_media_imdb entre 0 e 10",
    df,
    (F.col("nota_media_imdb") < 0)
    | (F.col("nota_media_imdb") > 10)
)

add_dq_result(
    "silver.tb_metricas_engajamento",
    "popularidade não negativa",
    df,
    F.col("popularidade") < 0
)

add_dq_result(
    "silver.tb_metricas_engajamento",
    "qtd_votos_tmdb não negativa",
    df,
    F.col("qtd_votos_tmdb") < 0
)

add_dq_result(
    "silver.tb_metricas_engajamento",
    "qtd_votos_imdb não negativa",
    df,
    F.col("qtd_votos_imdb") < 0
)

# Avaliacoes_usuarios

df = spark.table(
    f"{silver_schema}.tb_avaliacoes_usuarios"
)

add_dq_result(
    "silver.tb_avaliacoes_usuarios",
    "nota_usuario entre 0 e 10",
    df,
    (F.col("nota_usuario") < 0)
    | (F.col("nota_usuario") > 10)
)

add_dq_result(
    "silver.tb_avaliacoes_usuarios",
    "comentario_usuario preenchido",
    df,
    F.col("comentario_usuario").isNull()
    | (F.trim(F.col("comentario_usuario")) == "")
)

duplicados = (
    df
    .groupBy(
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dq_results.append({
    "table_name": "silver.tb_avaliacoes_usuarios",
    "check_name": "avaliação única",
    "total_rows": df.count(),
    "failed_rows": duplicados,
    "passed": duplicados == 0,
    "checked_at": datetime.now()
})

# Generos

df = spark.table(
    f"{silver_schema}.tb_generos"
)

generos_validos = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western"
]

add_dq_result(
    "silver.tb_generos",
    "genero em domínio válido",
    df,
    ~F.col("genero").isin(generos_validos)
)

duplicados = (
    df
    .groupBy("id_filme", "genero")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dq_results.append({
    "table_name": "silver.tb_generos",
    "check_name": "relação filme-gênero única",
    "total_rows": df.count(),
    "failed_rows": duplicados,
    "passed": duplicados == 0,
    "checked_at": datetime.now()
})


# Pessoas_empresas

df = spark.table(
    f"{silver_schema}.tb_pessoas_empresas"
)

tipos_validos = [
    "Ator",
    "Diretor",
    "Roteirista",
    "Produtora"
]

add_dq_result(
    "silver.tb_pessoas_empresas",
    "tipo_entidade em domínio válido",
    df,
    ~F.col("tipo_entidade").isin(tipos_validos)
)

add_dq_result(
    "silver.tb_pessoas_empresas",
    "nome_entidade preenchido",
    df,
    F.col("nome_entidade").isNull()
    | (F.trim(F.col("nome_entidade")) == "")
)

duplicados = (
    df
    .groupBy(
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dq_results.append({
    "table_name": "silver.tb_pessoas_empresas",
    "check_name": "relação filme-entidade única",
    "total_rows": df.count(),
    "failed_rows": duplicados,
    "passed": duplicados == 0,
    "checked_at": datetime.now()
})

# Cotacao_dolar

df = spark.table(
    f"{silver_schema}.tb_cotacao_dolar"
)

add_dq_result(
    "silver.tb_cotacao_dolar",
    "cotacao_compra não nula",
    df,
    F.col("cotacao_compra").isNull()
)

duplicados = (
    df
    .groupBy("data_cotacao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dq_results.append({
    "table_name": "silver.tb_cotacao_dolar",
    "check_name": "data_cotacao única",
    "total_rows": df.count(),
    "failed_rows": duplicados,
    "passed": duplicados == 0,
    "checked_at": datetime.now()
})

data_inicial = df.agg(
    F.min("data_cotacao")
).first()[0]

data_final = df.agg(
    F.max("data_cotacao")
).first()[0]

dias_esperados = (
    data_final - data_inicial
).days + 1

dias_existentes = (
    df
    .select("data_cotacao")
    .distinct()
    .count()
)

dias_ausentes = (
    dias_esperados - dias_existentes
)

dq_results.append({
    "table_name": "silver.tb_cotacao_dolar",
    "check_name": "série temporal contínua",
    "total_rows": dias_esperados,
    "failed_rows": dias_ausentes,
    "passed": dias_ausentes == 0,
    "checked_at": datetime.now()
})

# Df com os logs
df_dq_log = spark.createDataFrame(dq_results)

(
    df_dq_log.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{silver_schema}.dq_log")
)

display(
    spark.table(f"{silver_schema}.dq_log")
    .select(
        "table_name",
        "check_name",
        "total_rows",
        "failed_rows",
        "passed",
        "checked_at"
    )
    .orderBy(F.col("checked_at").desc())
)

table_name,check_name,total_rows,failed_rows,passed,checked_at
silver.tb_cotacao_dolar,série temporal contínua,8,0,true,2026-09-20T16:45:22.258Z
silver.tb_cotacao_dolar,data_cotacao única,8,0,true,2026-09-20T16:45:21.140Z
silver.tb_cotacao_dolar,cotacao_compra não nula,8,0,true,2026-09-20T16:45:20.413Z
silver.tb_pessoas_empresas,relação filme-entidade única,897550,0,true,2026-09-20T16:45:19.633Z
silver.tb_pessoas_empresas,nome_entidade preenchido,897550,0,true,2026-09-20T16:45:18.798Z
silver.tb_pessoas_empresas,tipo_entidade em domínio válido,897550,0,true,2026-09-20T16:45:18.042Z
silver.tb_generos,relação filme-gênero única,141965,0,true,2026-09-20T16:45:17.356Z
silver.tb_generos,genero em domínio válido,141965,0,true,2026-09-20T16:45:16.620Z
silver.tb_avaliacoes_usuarios,avaliação única,32412,0,true,2026-09-20T16:45:15.880Z
silver.tb_avaliacoes_usuarios,comentario_usuario preenchido,32412,0,true,2026-09-20T16:45:15.125Z
